In order to run this NB (=notebook), one needs to have a HuggingFace account and Token (say: "nimportequel").  Put the following into file .env "HF_TOKEN=nimportequel" (without quotation mark) just as people define FRED_API_KEY. Then one also needs to set JupyterServer to our project .venv. Below is a brief description:
Click the kernel name in the top-right of the notebook.
It’s the only clickable item on the right side of the toolbar.

A dialog will appear:
“Select Kernel”

Choose:
“Select Another Kernel…”

Choose:
“Python Environment…”  
or
“Enter interpreter path…”

Choose "+Create Python Environment"

Choose "Enter Interpreter Path ... Browse and Select a Python interpreter from anywhere" 
(in my coder workspace, this item is the last option being listed)
Paste your full interpreter path:

/home/coder/agentic-forecasting-c1-bmo-one/implementations/BAA10Y_forecasting/.venv/bin/python


In [ ]:
#%pip install langchain langchain-community langchain-huggingface sentence-transformers  pypdf rank_bm25

Run the following to see if access to HF is OK.

In [1]:
# @title
from dotenv import load_dotenv
import os
from huggingface_hub import notebook_login, login
#from google.colab import userdata

try:
    #hf_token = userdata.get('HF_TOKEN')
    load_dotenv()  # walks up from cwd to find the nearest .env
    hf_token = os.environ.get("HF_TOKEN")
    login(token=hf_token)
    print("Successfully logged in to Hugging Face.")
except Exception as e:
    print(f"Error logging in to Hugging Face: {e}")
    print("Please ensure you have added your Hugging Face token to Colab Secrets as 'HF_TOKEN'.")

/home/coder/agentic-forecasting-c1-bmo-one/implementations/BAA10Y_forecasting/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Successfully logged in to Hugging Face.


For the moment, ensure that only "intfloat/e5-base" is selected.

In [2]:
# @title Configure Embedding Model
EMBEDDING_MODEL_NAME = "intfloat/e5-base" # @param ["intfloat/e5-base", "FinanceMTEB/fin-e5-base", "all-MiniLM-L6-v2"]
MODEL_KWARGS = {} # @param {type:"raw"}
E5_MODEL_NAMES = {"intfloat/e5-base", "FinanceMTEB/fin-e5-base"}

#Do NOT use in bootcamp because of potential license issue
# If using FinanceMTEB/fin-e5-base, it might require trust_remote_code=True
if EMBEDDING_MODEL_NAME == "FinanceMTEB/fin-e5-base":
    MODEL_KWARGS = {'trust_remote_code': True}

print(f"Embedding model selected: {EMBEDDING_MODEL_NAME}")
print(f"Model kwargs: {MODEL_KWARGS}")

Embedding model selected: intfloat/e5-base
Model kwargs: {}


In [3]:
#v001
from langchain_huggingface import HuggingFaceEmbeddings
from typing import List

class E5Embeddings(HuggingFaceEmbeddings):
    """HuggingFaceEmbeddings + E5's required 'query: '/'passage: ' prefixes.

    E5-family models (intfloat/e5-*, FinanceMTEB/fin-e5-*) are trained with
    these prefixes; omitting them degrades retrieval. Only use this wrapper
    for E5-family models — all-MiniLM-L6-v2 was not trained with this
    convention and should be embedded unprefixed.
    """

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return super().embed_documents([f"passage: {t}" for t in texts])

    def embed_query(self, text: str) -> List[float]:
        return super().embed_query(f"query: {text}")

In [ ]:
#%pip install chromadb

Use the immediately following RAG to create vector store because it uses an advanced embedding model pretrained for financial sector. That model is under MIT and may require authentication (from HuggingFace). Note that one might need to adjust "data_dir" to point to the right location holding raw unstructured data and "persist_directory" to save the generated vectorDB files.

In [4]:
# 1. Force upgrade to modern library structures
#%pip install -q --upgrade langchain langchain-community langchain-huggingface langchain-text-splitters chromadb langchain-chroma openai-whisper unstructured python-pptx docx2txt pycd ..pdf

import os
import re
import shutil # Import shutil for directory removal
import whisper
import chromadb # Explicitly import chromadb for client creation
import datetime # Import datetime for date parsing
from collections import defaultdict
from typing import List
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, UnstructuredPowerPointLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Updated path to /content/gdrive
persist_directory = './chroma_db_big'

# Remove the existing Chroma DB directory to clear old embeddings if dimensions mismatch
if os.path.exists(persist_directory):
    print(f"Removing existing Chroma DB at {persist_directory} due to potential database corruption or dimension mismatch.")
    shutil.rmtree(persist_directory)

os.makedirs(persist_directory, exist_ok=True)

def extract_date_from_filename(filename):
    match = re.search(r'(\d{8})', filename)
    # If YYYYMMDD is found, append '0000' for HHMM, otherwise None
    return int(f"{match.group(1)}0000") if match else None

# New function to extract date and time from document content
def extract_datetime_from_content(text):
    # Pattern 1: "As adopted effective January 24, 2012" -> YYYYMMDD0000
    match_adopted = re.search(r"As adopted effective (January|February|March|April|May|June|July|August|September|October|November|December)\s+(\d{1,2}),\s+(\d{4})", text)
    if match_adopted:
        month_name, day, year = match_adopted.groups()
        month_num = datetime.datetime.strptime(month_name, '%B').month
        return int(f"{year}{month_num:02d}{int(day):02d}0000")

    # Pattern 2: "For release at 2 p.m. EST January 27, 2021" -> YYYYMMDDHHMM
    # Handles various timezones (EST, EDT, CST, CDT, MST, MDT, PST, PDT)
    match_release = re.search(r"For release at (\d{1,2})\s*(a\.m\.|p\.m\.)\s*(EST|EDT|CST|CDT|MST|MDT|PST|PDT)?\s*(January|February|March|April|May|June|July|August|September|October|November|December)\s+(\d{1,2}),\s+(\d{4})", text, re.IGNORECASE)
    if match_release:
        hour_str, ampm, _, month_name, day, year = match_release.groups()
        hour = int(hour_str)
        if ampm.lower() == 'p.m.' and hour != 12:
            hour += 12
        elif ampm.lower() == 'a.m.' and hour == 12: # 12 a.m. is midnight
            hour = 0
        month_num = datetime.datetime.strptime(month_name, '%B').month
        # Minutes are not captured, assume 00
        return int(f"{year}{month_num:02d}{int(day):02d}{hour:02d}00")

    return None

def load_audio(file_path):
    model = whisper.load_model("base")
    result = model.transcribe(file_path)
    return [Document(page_content=result["text"], metadata={"source": file_path})]

# ---------------------------------------------------------------------------
# Semantic chunking: section headers -> whole paragraph/bullet blocks -> pack
# ---------------------------------------------------------------------------
# Replaces fixed-length RecursiveCharacterTextSplitter slicing. Guarantees:
#   1. A chunk never spans two different named sections.
#   2. A chunk is built from whole blocks (paragraphs, or a whole contiguous
#      bullet list) -- never split mid-sentence or mid-bullet-list.
#   3. Blocks are packed greedily into paragraph GROUPS up to max_chars,
#      rather than emitting one chunk per paragraph.

SECTION_HEADERS = [
    "Financial Conditions",
    "Credit Markets",
    "Monetary Policy",
    "Liquidity and Market Functioning",
    "Risks to the Outlook",
]
_HEADER_PATTERN = re.compile(r"(?m)^(" + "|".join(re.escape(h) for h in SECTION_HEADERS) + r")\s*$")
_BULLET_PREFIXES = ("•", "- ", "o ", "* ")


def _split_into_blocks(text: str) -> List[str]:
    """Paragraphs as atomic blocks; a run of consecutive bullet lines stays one block."""
    raw_blocks = re.split(r"\n\s*\n", text.strip())
    blocks, buffer = [], ""
    for raw in raw_blocks:
        raw = raw.strip()
        if not raw:
            continue
        is_bullet = raw.lstrip().startswith(_BULLET_PREFIXES)
        if is_bullet and buffer.lstrip().startswith(_BULLET_PREFIXES):
            buffer += "\n" + raw  # same bullet list continuing -- merge, don't split
        else:
            if buffer:
                blocks.append(buffer)
            buffer = raw
    if buffer:
        blocks.append(buffer)
    return blocks


def _split_sentence_safe(block: str, max_chars: int) -> List[str]:
    """Fallback ONLY for a single block that alone exceeds max_chars. Naive regex --
    flag long blocks for manual review rather than trusting this on abbreviation-heavy
    Fed prose (e.g. 'U.S.'); nltk/spacy sentence tokenizers are sturdier if this matters."""
    sentences = re.split(r"(?<=[.!?])\s+(?=[A-Z])", block)
    chunks, current = [], ""
    for sent in sentences:
        if current and len(current) + len(sent) + 1 > max_chars:
            chunks.append(current)
            current = sent
        else:
            current = f"{current} {sent}".strip()
    if current:
        chunks.append(current)
    return chunks


def semantic_chunk_document(text: str, max_chars: int = 1200) -> List[dict]:
    """Section-header split -> whole-block packing -> sentence-safe fallback for oversized blocks."""
    matches = list(_HEADER_PATTERN.finditer(text))
    if not matches:
        sections = [("unlabeled", text)]  # e.g. FOMC statements have none of these headers
    else:
        sections = []
        if matches[0].start() > 0:
            sections.append(("preamble", text[: matches[0].start()]))
        for i, m in enumerate(matches):
            end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
            sections.append((m.group(1), text[m.end():end]))

    results: List[dict] = []
    for section_name, section_text in sections:
        current = ""
        for block in _split_into_blocks(section_text):
            if len(block) > max_chars:
                if current:
                    results.append({"section": section_name, "text": current})
                    current = ""
                results.extend({"section": section_name, "text": p} for p in _split_sentence_safe(block, max_chars))
                continue
            if current and len(current) + len(block) + 2 > max_chars:
                results.append({"section": section_name, "text": current})
                current = block
            else:
                current = f"{current}\n\n{block}".strip()
        if current:
            results.append({"section": section_name, "text": current})
    return results

raw_docs = []
data_dir = './unstructured_data'  #need to adjust/customize the directory for the raw files location
supported_exts = ('.pdf', '.docx', '.pptx', '.mp3', '.wav')
files = [f for f in os.listdir(data_dir) if f.lower().endswith(supported_exts)]
print(f"Processing {len(files)} files...")

for file in files:
    file_path = os.path.join(data_dir, file)
    filename_date_val = extract_date_from_filename(file) # Existing filename-based date as fallback

    try:
        file_docs = []
        extracted_content_date = None

        if file.lower().endswith('.pdf'):
            file_docs = PyPDFLoader(file_path).load()
            # Attempt to extract date from the first page's content for PDFs
            if file_docs and file_docs[0].page_content:
                extracted_content_date = extract_datetime_from_content(file_docs[0].page_content)
        elif file.lower().endswith('.docx'):
            file_docs = Docx2txtLoader(file_path).load()
        elif file.lower().endswith('.pptx'):
            file_docs = UnstructuredPowerPointLoader(file_path).load()
        elif file.lower().endswith(('.mp3', '.wav')):
            file_docs = load_audio(file_path)

        for d in file_docs:
            # Prioritize content-extracted date, fall back to filename date if not found
            if extracted_content_date is not None:
                d.metadata['file_date'] = extracted_content_date
            elif filename_date_val is not None:
                d.metadata['file_date'] = filename_date_val
            else:
                d.metadata['file_date'] = None # No date found from either source

            d.metadata['source_file'] = file
        raw_docs.extend(file_docs)
    except Exception as e:
        print(f"Error processing {file}: {e}")

if raw_docs:
    # Semantic chunking (section headers -> paragraph/bullet groups -> sentence-safe
    # fallback) replaces fixed-length slicing. raw_docs is one Document per PDF PAGE
    # (PyPDFLoader) -- merge pages per source file first so a section spanning a page
    # break, or a bullet list split across pages, isn't cut there.
    pages_by_file = defaultdict(list)
    for d in raw_docs:
        pages_by_file[d.metadata["source_file"]].append(d)

    splits: List[Document] = []
    for source_file, pages in pages_by_file.items():
        full_text = "\n\n".join(p.page_content for p in pages)
        file_meta = {"file_date": pages[0].metadata.get("file_date"), "source_file": source_file}
        for chunk in semantic_chunk_document(full_text, max_chars=1200):
            splits.append(Document(page_content=chunk["text"], metadata={**file_meta, "section": chunk["section"]}))

    print(f"Produced {len(splits)} semantic chunks across {len(pages_by_file)} documents.")

    #embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME, model_kwargs=MODEL_KWARGS) # Using configured model
    #20260808
    if EMBEDDING_MODEL_NAME in E5_MODEL_NAMES:
        embeddings = E5Embeddings(model_name=EMBEDDING_MODEL_NAME, model_kwargs=MODEL_KWARGS)
    else:
        embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME, model_kwargs=MODEL_KWARGS)

    # Explicitly create the ChromaDB client
    try:
        client = chromadb.PersistentClient(path=persist_directory)
        print(f"ChromaDB PersistentClient successfully created at {persist_directory}.")
    except Exception as e:
        print(f"Error creating ChromaDB PersistentClient: {e}")
        print("This often indicates a deeper issue with ChromaDB's installation or interaction with the file system.")
        raise # Re-raise the exception to stop execution if client cannot be initialized

    vectorstore = Chroma.from_documents(
        documents=splits,
        embedding=embeddings,
        client=client, # Pass the explicit client instance
        collection_name="my_rag_collection_e5_base" # Explicitly name the collection for a fresh start
    )
    print(f"Successfully indexed {len(splits)} chunks and saved to {persist_directory}.")
else:
    print("No documents found to index.")

/tmp/ipykernel_6928/1568176166.py:13: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, UnstructuredPowerPointLoader


Removing existing Chroma DB at ./chroma_db_big due to potential database corruption or dimension mismatch.
Processing 2 files...
Produced 12 semantic chunks across 2 documents.


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3511.68it/s]


ChromaDB PersistentClient successfully created at ./chroma_db_big.
Successfully indexed 12 chunks and saved to ./chroma_db_big.


In [1]:
#Diagnostic
import sys
print(sys.executable)

!uv pip show openai-whisper


/home/coder/agentic-forecasting-c1-bmo-one/implementations/BAA10Y_forecasting/.venv/bin/python
Name: openai-whisper
Version: 20250625
Location: /home/coder/agentic-forecasting-c1-bmo-one/implementations/BAA10Y_forecasting/.venv/lib/python3.12/site-packages
Requires: more-itertools, numba, numpy, tiktoken, torch, tqdm, triton
Required-by:


In [5]:
def normalize_date_filter_value(date_val, is_before):
    """
    Normalizes a date filter value to YYYYMMDDHHMM format.
    If date_val is YYYYMMDD, it becomes YYYYMMDD0000 (start of day) or YYYYMMDD2359 (end of day).
    If date_val is YYYYMMDDHHMM, it's used as is.
    """
    if date_val is None:
        return None
    date_str = str(date_val)
    if len(date_str) == 8: # YYYYMMDD format
        if is_before:
            return int(date_str + "2359") # End of day for upper bound
        else:
            return int(date_str + "0000") # Beginning of day for lower bound
    elif len(date_str) == 12: # YYYYMMDDHHMM format
        return int(date_str)
    return None # Invalid format

def query_rag_with_date_filter(query, date_before=None, date_after=None, threshold=0.4):
    """
    Query Chroma with optional date restrictions.
    Example: date_before=20260312, date_after=20250101
    """
    search_kwargs = {"k": 3}
    filters = []

    normalized_date_after = normalize_date_filter_value(date_after, is_before=False)
    normalized_date_before = normalize_date_filter_value(date_before, is_before=True)

    if normalized_date_after:
        filters.append({"file_date": {"$gte": normalized_date_after}})
    if normalized_date_before:
        filters.append({"file_date": {"$lte": normalized_date_before}})

    if filters:
        if len(filters) == 1:
            search_kwargs["filter"] = filters[0]
        else:
            search_kwargs["filter"] = {"$and": filters}

    print(f"[DEBUG] Query: '{query}'")
    print(f"[DEBUG] Search arguments for ChromaDB: {search_kwargs}")

    # Get results with relevance scores (0 to 1 range)
    results = vectorstore.similarity_search_with_relevance_scores(query, **search_kwargs)

    print(f"[DEBUG] Raw results from vectorstore: {results}")

    if not results:
        return [], 0.0, True

    avg_score = sum(score for doc, score in results) / len(results)
    needs_web_search = avg_score < threshold

    print(f"Confidence: {avg_score:.4f} | Needs Web Search: {needs_web_search}")
    for doc, score in results:
        print(f"- [Date: {doc.metadata.get('file_date')}] {doc.metadata.get('source_file')}")

    return results, avg_score, needs_web_search

# Example Demo:
# results, conf, search = query_rag_with_date_filter("inflation outlook", date_before=20260312)

In [6]:
from langchain_community.retrievers import BM25Retriever

# Initialize BM25 on the same splits used for Chroma
bm25_retriever = BM25Retriever.from_documents(splits)
bm25_retriever.k = 3

print("BM25 Document DB initialized.")

# Example usage of the date-filtered search:
# This will find documents strictly before 
query_text = "holdings of Treasury securities"
#"target range for the federal funds rate"
target_date = 202509180001

results, confidence, needs_search = query_rag_with_date_filter(query_text, date_before=target_date, date_after=None)

print(f"[DEBUG] Raw results from vectorstore: {results}")
if needs_search:
    print("\n[!] Local confidence is low. Recommendation: Search Google/Web for more info.")
else:
    print("\n[+] Local confidence is sufficient.")

BM25 Document DB initialized.
[DEBUG] Query: 'holdings of Treasury securities'
[DEBUG] Search arguments for ChromaDB: {'k': 3, 'filter': {'file_date': {'$lte': 202509180001}}}
[DEBUG] Raw results from vectorstore: [(Document(metadata={'source_file': 'monetary20250917a1.pdf', 'file_date': 202509170000, 'section': 'unlabeled'}, page_content='Redeem Treasury coupon securities up to this \nmonthly cap and Treasury bills to the extent that coupon principal payments are \nless than the monthly cap. \no Reinvest the amount of principal payments from the Federal Reserve\'s holdings of \nagency debt and agency mortgage-backed securities (MBS) received in each \ncalendar month that exceeds a cap of $35 billion per month into Treasury \nsecurities to roughly match the maturity composition of Treasury securities \noutstanding. \no Allow modest deviations from stated amounts for reinvestments, if needed for \noperational reasons."'), 0.7610264417125636), (Document(metadata={'section': 'unlabeled', 

In [7]:
# @title
#%pip install -q -U langchain-chroma langchain-community langchain-core rank_bm25 langchain-huggingface
#build both VectorDB search and document search with BM25.
import os
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_core.retrievers import BaseRetriever
from typing import List
from langchain_core.documents import Document

# Fallback implementation if EnsembleRetriever cannot be imported
try:
    from langchain_community.retrievers import EnsembleRetriever
except (ImportError, ModuleNotFoundError):
    class EnsembleRetriever(BaseRetriever):
        retrievers: List[BaseRetriever]
        weights: List[float]
        def _get_relevant_documents(self, query: str, **kwargs) -> List[Document]:
            all_docs = []
            for retriever in self.retrievers:
                all_docs.extend(retriever.get_relevant_documents(query))
            return all_docs[:5]

if 'embeddings' not in locals():
    embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME, model_kwargs=MODEL_KWARGS)

persist_directory = './chroma_db_big'

try:
    vectorstore = Chroma(
        persist_directory=persist_directory,
        embedding_function=embeddings,
        collection_name="my_rag_collection_e5_base" # Added collection name
    )
    count = vectorstore._collection.count()
    print(f"Successfully connected to Chroma. Found {count} chunks.")
except Exception as e:
    print(f"Error connecting to database: {e}")

if 'vectorstore' in locals():
    chroma_retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

    if 'splits' in locals() and splits:
        bm25_retriever = BM25Retriever.from_documents(splits)
        bm25_retriever.k = 3
        ensemble_retriever = EnsembleRetriever(retrievers=[bm25_retriever, chroma_retriever], weights=[0.5, 0.5])
        print("Hybrid Retrieval (BM25 + Chroma) is now fully ACTIVE.")
    else:
        ensemble_retriever = chroma_retriever
        print("Chroma Vector Search is ACTIVE. (BM25 requires 'splits' in memory)")

Successfully connected to Chroma. Found 12 chunks.
Hybrid Retrieval (BM25 + Chroma) is now fully ACTIVE.


In [8]:

# @title
from langchain_community.retrievers import BM25Retriever

# Robust import for EnsembleRetriever
try:
    from langchain.retrievers import EnsembleRetriever
except ImportError:
    try:
        from langchain_community.retrievers.ensemble import EnsembleRetriever
    except ImportError:
        # Simple implementation if the class is completely unavailable in the installed version
        class EnsembleRetriever:
            def __init__(self, retrievers, weights):
                self.retrievers = retrievers
                self.weights = weights
            def invoke(self, query):
                return self.retrievers[0].invoke(query)

# Initialize Retrievers
chroma_retriever = vectorstore.as_retriever(search_kwargs={'k': 3})
bm25_retriever = BM25Retriever.from_documents(splits)
bm25_retriever.k = 3

# Create Hybrid Ensemble Retriever
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, chroma_retriever],
    weights=[0.5, 0.5]
)

print("Hybrid EnsembleRetriever initialized successfully.")

Hybrid EnsembleRetriever initialized successfully.


In [9]:
def normalize_date_filter_value(date_val, is_before):
    """
    Normalizes a date filter value to YYYYMMDDHHMM format.
    If date_val is YYYYMMDD, it becomes YYYYMMDD0000 (start of day) or YYYYMMDD2359 (end of day).
    If date_val is YYYYMMDDHHMM, it's used as is.
    """
    if date_val is None:
        return None
    date_str = str(date_val)
    if len(date_str) == 8: # YYYYMMDD format
        if is_before:
            return int(date_str + "2359") # End of day for upper bound
        else:
            return int(date_str + "0000") # Beginning of day for lower bound
    elif len(date_str) == 12: # YYYYMMDDHHMM format
        return int(date_str)
    return None # Invalid format

def query_hybrid_rag(query, date_before=None, date_after=None, threshold=0.4):
    """
    Queries the Hybrid Ensemble Retriever and calculates confidence.
    """
    # Note: EnsembleRetriever typically returns documents without scores by default.
    # To provide the confidence metric you requested, we use the base Chroma retriever
    # for the score while using the ensemble for the document set.
    search_kwargs = {"k": 3}
    filters = []

    normalized_date_after = normalize_date_filter_value(date_after, is_before=False)
    normalized_date_before = normalize_date_filter_value(date_before, is_before=True)

    if normalized_date_after:
        filters.append({"file_date": {"$gte": normalized_date_after}})
    if normalized_date_before:
        filters.append({"file_date": {"$lte": normalized_date_before}})

    if filters:
        if len(filters) == 1:
            search_kwargs["filter"] = filters[0]
        else:
            search_kwargs["filter"] = {"$and": filters}

    # Get ensemble results
    docs = ensemble_retriever.invoke(query)

    # Get scores specifically from Chroma for the confidence metric
    results_with_scores = vectorstore.similarity_search_with_relevance_scores(query, **search_kwargs)

    avg_score = sum(score for doc, score in results_with_scores) / len(results_with_scores) if results_with_scores else 0.0
    needs_web_search = avg_score < threshold

    print(f"Hybrid Query: {query}")
    print(f"Confidence Score: {avg_score:.4f}")
    print(f"Action: {'TRIGGER WEB SEARCH' if needs_web_search else 'LOCAL DATA SUFFICIENT'}")

    return docs, avg_score, needs_web_search

# Example usage:
# docs, conf, search = query_hybrid_rag("inflation trends", date_before=20260101, date_after=20250101)

In [10]:
docs, conf, search = query_hybrid_rag("Increase the System Open Market Account holdings of Treasury securities by how much", date_before=202601010000)
print(docs)

Hybrid Query: Increase the System Open Market Account holdings of Treasury securities by how much
Confidence Score: 0.7628
Action: LOCAL DATA SUFFICIENT
[Document(metadata={'file_date': 202509170000, 'source_file': 'monetary20250917a1.pdf', 'section': 'unlabeled'}, page_content='Decisions Regarding Monetary Policy Implementation \nThe Federal Reserve has made the following decisions to implement the monetary policy stance \nannounced by the Federal Open Market Committee in its statement on September 17, 2025: \n• The Board of Governors of the Federal Reserve System voted to lower the interest rate paid \non reserve balances to 4.15 percent, effective September 18, 2025. \n• As part of its policy decision, the Federal Open Market Committee voted to direct the Open \nMarket Desk at the Federal Reserve Bank of New York, until instructed otherwise, to \nexecute transactions in the System Open Market Account in accordance with the following \ndomestic policy directive: \n"Effective Septembe

In [11]:
# Extract and print the content of each document
from datetime import datetime

print(f"\n--- Extracted Document Contents ({datetime.now().isoformat(timespec='seconds')}) ---")
for i, doc in enumerate(docs):
    print(f"Document {i+1} (Source: {doc.metadata.get('source_file')} | Date: {doc.metadata.get('file_date')}):")
    print(doc.page_content)
    print("\n" + "-" * 30 + "\n")


--- Extracted Document Contents (2026-09-08T02:40:08) ---
Document 1 (Source: monetary20250917a1.pdf | Date: 202509170000):
Decisions Regarding Monetary Policy Implementation 
The Federal Reserve has made the following decisions to implement the monetary policy stance 
announced by the Federal Open Market Committee in its statement on September 17, 2025: 
• The Board of Governors of the Federal Reserve System voted to lower the interest rate paid 
on reserve balances to 4.15 percent, effective September 18, 2025. 
• As part of its policy decision, the Federal Open Market Committee voted to direct the Open 
Market Desk at the Federal Reserve Bank of New York, until instructed otherwise, to 
execute transactions in the System Open Market Account in accordance with the following 
domestic policy directive: 
"Effective September 18, 2025, the Federal Open Market Committee directs the Desk to: 
o Undertake open market operations as necessary to maintain the federal funds rate 
in a target 